# 🧠 DistilBERT Fine-Tuning on GoEmotions (Ekman Taxonomy)
### AI-Based Employee Wellness & Emotion Detection Platform (MoodMentor)
This notebook trains a DistilBERT sequence classifier on the GoEmotions Ekman dataset, maps probabilities with `prob > 0.5`, and computes `f1_micro` and `f1_macro` metrics.

## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install transformers datasets torch scikit-learn accelerate

## 📥 Step 2: Load the GoEmotions Dataset

In [ ]:
from datasets import load_dataset

# Load GoEmotions simplified_ekman subset
dataset = load_dataset("AiLab-IMCS-UL/go_emotions-en", "simplified_ekman")
print(dataset)

## 🔍 Step 3: Inspect the Schema

In [ ]:
print("Columns:", dataset["train"].column_names)
print("Sample row:", dataset["train"][0])

## 🏷️ Step 4: Define Emotion Labels (Ekman Taxonomy)
Mapping the 7 basic Ekman classes: `anger`, `disgust`, `fear`, `joy`, `sadness`, `surprise`, `neutral`

In [ ]:
emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
id2label = {i: label for i, label in enumerate(emotions)}
label2id = {label: i for i, label in enumerate(emotions)}

print("Emotion mapping:", id2label)

## 🔤 Step 5: Tokenize the Dataset (Convert Text to Numerical Identifiers)

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(examples):
    labels = [int(lbl[0]) if len(lbl) > 0 else 6 for lbl in examples["labels_ekman"]]
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=96)
    tokens["labels"] = labels
    return tokens

# Select subset for fast training in Colab (adjust range as needed)
train_data = dataset["train"].select(range(3000))
val_data = dataset["validation"].select(range(500))

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)
print("Tokenization complete!")

## 🧠 Step 6: Configure DistilBERT Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(emotions),
    id2label=id2label,
    label2id=label2id
)
print("DistilBERT configured with 7 emotion output heads.")

## 📊 Step 7: Train Model with Micro & Macro F1 Evaluation Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=20
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

## 🎯 Step 8: Inference with Threshold Mapping (`prob > 0.5`)

In [ ]:
import torch

text = "I am so proud of our team, we did an amazing job today!"
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).squeeze().tolist()

print("Input Text:", text)
print("-" * 50)

# Threshold and map to labels (prob > 0.5)
for i, prob in enumerate(probs):
    if prob > 0.5:
        print(f"{emotions[i]}: {prob:.2f}")

print("\n## Evaluation Metric")
print("During training, micro and macro **F1 scores** were used:")
print("- 'f1_micro': high recall on rare labels")
print("- 'f1_macro': balanced performance across labels")